In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
df = pd.read_csv('../data/processed/df_processed.csv')
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week,month,hour
0,BADF67E2C5058F19,classic_bike,2025-05-11 17:22:39.471,2025-05-11 18:11:19.249,DuSable Lake Shore Dr & North Blvd,LF-005,Winthrop Ave & Lawrence Ave,TA1308000021,41.911722,-87.626804,41.968812,-87.657659,member,0 days 00:48:39.778000,Sunday,May,17
1,0210AE485D59C8C5,electric_bike,2025-05-05 08:02:09.251,2025-05-05 08:12:07.549,Damen Ave & Grand Ave,TA1308000006,Desplaines St & Jackson Blvd,15539,41.892394,-87.676885,41.878119,-87.643948,member,0 days 00:09:58.298000,Monday,May,8
2,5E68FE5B9283E4C4,classic_bike,2025-05-02 10:32:33.062,2025-05-02 10:39:07.262,LaSalle St & Illinois St,13430,Clark St & Elm St,TA1307000039,41.890762,-87.631697,41.903322,-87.632999,member,0 days 00:06:34.200000,Friday,May,10
3,13D2DCD6FB872858,classic_bike,2025-05-12 11:12:16.579,2025-05-12 11:17:25.126,Milwaukee Ave & Rockwell St,13242,Damen Ave & Cortland St,13133,41.920330,-87.693090,41.915983,-87.677335,member,0 days 00:05:08.547000,Monday,May,11
4,F04DF9EE163351DD,classic_bike,2025-05-01 10:13:36.821,2025-05-01 10:17:40.548,Halsted St & Roosevelt Rd,TA1305000017,Clinton St & Roosevelt Rd,WL-008,41.867324,-87.648625,41.867118,-87.641088,member,0 days 00:04:03.727000,Thursday,May,10


In [3]:
df.isna().sum()

ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name    704125
start_station_id      704125
end_station_name      703035
end_station_id        703035
start_lat                  0
start_lng                  0
end_lat                    0
end_lng                    0
member_casual              0
ride_length                0
day_of_week                0
month                      0
hour                       0
dtype: int64

In [4]:
df.shape

(3225974, 17)

Categorical Distribution 

In [98]:
# Bike type
bike_counts = df['rideable_type'].value_counts().reset_index()
bike_counts.columns = ['rideable_type', 'count']

In [99]:
# visual 
fig1 = px.bar(
    bike_counts, 
    x='rideable_type',
    y='count',
    text='count',
    title='Distribution of Bike Types', 
    width=600,
    height=400
)
fig1.show()

In [7]:
df['member_casual'].value_counts()

member_casual
member    1922429
casual    1303545
Name: count, dtype: int64

In [8]:
# visual 
count = df['member_casual'].value_counts().reset_index()
count.columns = ['member_casual', 'count']
fig2 = px.bar(
    count, 
    x='member_casual',
    y='count',
    text='count',
    title='Distribution of User Types', 
    width=600,
    height=400
    
)
fig2.show()

--- Usage Behavior --- 

when do casual users ride most? 

In [9]:
# Depend on day
casual_days = df[df['member_casual'] == 'casual']['day_of_week'].value_counts()
member_days = df[df['member_casual'] == 'member']['day_of_week'].value_counts()

In [10]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

counts = (
    df.groupby(['day_of_week', 'member_casual'])
      .size()
      .reset_index(name='ride_count')
)

fig = px.bar(
    counts,
    x='day_of_week',
    y='ride_count',
    color='member_casual',
    barmode='group',
    text='ride_count',
    category_orders={'day_of_week': day_order},
    title='Number of Rides by Day of Week', 
    width=600,
    height=400
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

-> Insight: casual riders use Cyclistic bikes significantly more during weekends, while members maintain more consistent usage throughout the week.


Weekend usage among casual riders is noticeably higher than among members, suggesting that casual riders primarily use bikes for leisure activities rather tahn commuting. This presents on opportunity to market annual memberships as a cost-effective for frequent recreeational riders. 

In [11]:
# Depend on hours
casual_hours = df[df['member_casual'] == 'casual']['hour'].value_counts()
member_hours = df[df['member_casual'] == 'member']['hour'].value_counts()

In [12]:
import plotly.express as px

hours = (
    df.groupby(['hour', 'member_casual'])
      .size()
      .reset_index(name='ride_count')
)

fig = px.bar(
    hours,
    x='hour',
    y='ride_count',
    color='member_casual',
    barmode='group',
    text='ride_count',
    title='Number of Rides by Hour of Day', 
    width=800, 
    height=500
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')

fig.update_layout(
    xaxis_title='Hour of Day',
    yaxis_title='Number of Rides'
)

fig.show()

-> Both casual riders and members reach peak usage during the late afternoon, particularly around 5 PM.

Both casual riders and members exhibit peak usage around 5 PM, suggesting that the service is commonly used after work or school. However casual riders are more concentrated on weekends and tend to take longer rides, indicating a more leisure-oriented usage pattern.

In [13]:
# Seasonal trend 
casual_months = df[df['member_casual'] == 'casual']['month'].value_counts()
member_months = df[df['member_casual'] == 'member']['month'].value_counts()

In [14]:
import plotly.express as px

months = (
    df.groupby(['month', 'member_casual'])
      .size()
      .reset_index(name='ride_count')
)

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

fig = px.bar(
    months,
    x='month',
    y='ride_count',
    color='member_casual',
    barmode='group',
    text='ride_count',
    category_orders={'month': month_order},
    title='Number of Rides by Month', 
    width=600, 
    height=400
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')

fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Number of Rides'
)

fig.show()

-> Both casual riders and members are most active during the warmer months, with peak usage occurring from May to September. 

Bike usage increases significantly during the warmer months, especially from May to September This seasonal trend suggests that Cyclistic should concentrate marketing campaigns during this period, when both casual riders and members are most active.

--- Ride Duration ---

Do casual users take longer rides than members ? 

What is the average ride length for each user type ? 


In [15]:
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

In [16]:
# Creating duration column
df['duration'] = df['ended_at'] - df['started_at']
df['duration'] = df['duration'].dt.total_seconds()/60

In [18]:
# check outliers 
Q1 = df['duration'].quantile(0.25)
Q3 = df['duration'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df[(df['duration']<lower) | (df['duration']>upper)]

outliers.shape

(231056, 18)

In [21]:
duration_without_outliers = df[
    (df['duration']>=lower) & (df['duration']<=upper)
]

In [30]:
duration_no_outliers = duration_without_outliers.groupby('member_casual')['duration'].mean().reset_index()

In [32]:
avg_duration = (duration_no_outliers)
fig = px.bar(
    avg_duration, 
    x='member_casual', 
    y='duration', 
    text='duration', 
    title='Average Ride Duration by User Type'
)
fig.update_traces(texttemplate='%{text:.2f} min', textposition='outside')

fig.show()

-> Casual riders take longer trips than members, with an average rie duration of 12.66 minutes compared to 10.82 minutes for members. This suggests that casual riders are more likely to use bikes for leisure purposes, while members tend to use them for shorter, routine trips. 

Are long rides associated with weekends or weekdays ? 

In [49]:
avg_duratioin_by_day = df.groupby('day_of_week')['duration'].median().sort_values(ascending = False).reset_index()
avg_duratioin_by_day

,day_of_week,duration
0,Saturday,11.816075
1,Sunday,11.682117
2,Friday,10.181550
3,Monday,9.736500
4,Thursday,9.710800
5,Tuesday,9.675100
6,Wednesday,9.611342


In [53]:
# visual 
fig = px.bar(
    avg_duratioin_by_day, 
    x='day_of_week', 
    y='duration', 
    text='duration', 
    title='Median Ride Duration by Day of Week'
)
fig.update_traces(texttemplate='%{text:.2f} min', textposition='outside')
fig.show()

-> Average ride duration is significantly higher on weekends, with Saturday and Sunday showing the longest trips. This suggests that users are more likely to take leisure-oriented rides during weekends, while weekday trips trend to be shorter and more utilitarian. 

--- Bike Type --- 

Do members and casual users use different bike types ? 

In [55]:
bike_type_by_user = df.groupby(['member_casual'])['rideable_type'].value_counts().reset_index(name='count')
bike_type_by_user

,member_casual,rideable_type,count
0,casual,electric_bike,841786
1,casual,classic_bike,461759
2,member,electric_bike,1219383
3,member,classic_bike,703046


In [56]:
fig = px.bar(
    bike_type_by_user, 
    x='rideable_type', 
    y='count', 
    color='member_casual', 
    barmode='group',
    text='count',
    title='Distribution of Bike Types by User Type'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

--- Location / Station --- 

In [59]:
df_station = pd.read_csv('../data/processed/df_without_nulls.csv')

In [60]:
df_station.isna().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
member_casual         0
ride_length           0
day_of_week           0
month                 0
hour                  0
dtype: int64

What are the most popular start stations for casual users ? 

In [76]:
top_casual_stations = (
    df[df['member_casual'] == 'casual']
    .groupby('start_station_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=True)
    .tail(10)
)
top_casual_stations

,start_station_name,count
739,Montrose Harbor,8828
317,Dusable Harbor,11205
1320,Shedd Aquarium,11555
1416,Theater on the Lake,12437
723,Millennium Park,13073
314,DuSable Lake Shore Dr & North Blvd,16069
716,Michigan Ave & Oak St,16857
764,Navy Pier,17867
1411,Streeter Dr & Grand Ave,19199
313,DuSable Lake Shore Dr & Monroe St,20932


In [84]:
fig = px.bar(
    top_casual_stations,
    x='count',
    y='start_station_name',
    orientation='h',
    hover_name='start_station_name',
    text='count', 
    width=600,
    height=400,
)

fig.update_yaxes(showticklabels=False)

fig.show()

--- Time-Based Patterns --- 

How does usage change across months ? 

In [100]:
monthly_usage = (
    df.groupby(['month', 'member_casual'])
    .size()
    .reset_index(name='count')
)
monthly_usage

,month,member_casual,count
0,April,casual,20
1,April,member,18
2,August,casual,337209
3,August,member,452342
4,July,casual,322502
5,July,member,439975
6,June,casual,291135
7,June,member,386705
8,May,casual,182275
9,May,member,319751


In [95]:
fig = px.line(
    monthly_usage,
    x='month',
    y='count',
    color='member_casual',
    markers=True,
    title='Monthly Usage Trend: Members vs Casual Riders'
)

fig.update_layout(template='simple_white')
fig.show()

Is there a seasonal trend in membership usage ? 

In [101]:
member_monthly = (
    df[df['member_casual'] == 'member']
    .groupby('month')
    .size()
    .reset_index(name='count')
    .sort_values('month')
)
member_monthly

,month,count
0,April,18
1,August,452342
2,July,439975
3,June,386705
4,May,319751
5,September,323638


In [97]:
fig = px.line(
    member_monthly,
    x='month',
    y='count',
    markers=True,
    title='Seasonal Trend in Member Usage'
)

fig.update_layout(template='simple_white')
fig.show()